# Price Elasticity of Liquor Demand in Iowa
## Demand Analytics — Final Project

---

## Research Questions

> **Main:** How sensitive is liquor demand to price changes across counties and product categories in Iowa?
>
> **Secondary:** Do holidays and weekends amplify or weaken that price sensitivity, and does the effect persist dynamically?

This notebook frames the problem as a **demand modeling problem** — demand is a function of price, market characteristics, and time structure. We estimate price elasticities using a log-log regression framework with high-dimensional fixed effects, and validate predictions on a held-out time period.

---

## Model Roadmap

| Group | Models | Core Question |
|---|---|---|
| Functional Form | M0 Linear, M1 Semi-log, M2 Log-log ⭐, M3 Quadratic | What shape best describes price-demand? |
| Heterogeneity (Broad) | M4 Category (8 types), M5 Urban/Rural, M6 Cat × Market | Who is price sensitive? |
| Heterogeneity (Granular) | M4b Granular categories, M5b County-level, M5c County × Category heatmap | Where exactly are pricing levers strongest? |
| Calendar | M7 Holiday Interaction, M8 Month × Price, M9 Year Trend | When does sensitivity change? |
| Dynamic | M10 Lagged Demand, M11 Lagged Price | Does past demand/price predict today? |
| Cross-Price | M12 Vodka ↔ Whiskey | Are spirits substitutes? |
| Validation | Holdout 2016–2017 | How well does the model predict out of sample? |

---
# 1. Setup

Install dependencies:
```
pip install pyfixest pandas numpy matplotlib seaborn statsmodels linearmodels scikit-learn
```

We use **`pyfixest`** as the primary modeling package for panel fixed-effects models (Python equivalent of R's `fixest`), and **`statsmodels`** for granular OLS models and holdout validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import re
import pyfixest as pf
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='tab10')

DATA_PATH = 'cleaned_iowa_liquor_sales_merged.csv'
print('All packages loaded.')

---
# 2. Data Loading & Preparation

The raw dataset contains **12.3 million** individual liquor transactions across 99 Iowa counties from 2012–2017.

In [ ]:
print('Loading raw data... (this may take ~30 seconds for 12M rows)')

df = pd.read_csv(DATA_PATH, low_memory=False)

# Parse dollar-formatted columns
for col in ['State Bottle Cost', 'State Bottle Retail', 'Sale (Dollars)']:
    df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# Date and time features
df['date']  = pd.to_datetime(df['Date'])
df['month'] = df['date'].dt.month
df['year']  = df['Year'].astype(int)
df['ym']    = df['date'].dt.to_period('M').astype(str)

# Weekend flag (Saturday=5, Sunday=6)
df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)

# Price per liter at transaction level
df['price_per_liter'] = df['State Bottle Retail'] / (df['Bottle Volume (ml)'] / 1000)
df = df[(df['price_per_liter'] > 0) & np.isfinite(df['price_per_liter'])]

print(f'Rows after cleaning: {len(df):,}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Counties: {df["County"].nunique()} | Raw categories: {df["Category Name"].nunique()}')

## 2.1 Category Consolidation

The raw data has 130 granular category names. We consolidate into **8 broad spirit categories** for the main elasticity models. We preserve the granular categories for the deep-dive analysis in Section 7.

In [ ]:
cat_map = {
    '100 PROOF VODKA': 'Vodka', 'VODKA 80 PROOF': 'Vodka',
    'VODKA FLAVORED': 'Vodka', 'IMPORTED VODKA': 'Vodka',
    'IMPORTED VODKA - MISC': 'Vodka', 'LOW PROOF VODKA': 'Vodka',
    'OTHER PROOF VODKA': 'Vodka',
    'BLENDED WHISKIES': 'Whiskey', 'CANADIAN WHISKIES': 'Whiskey',
    'IRISH WHISKIES': 'Whiskey', 'SCOTCH WHISKIES': 'Whiskey',
    'SINGLE MALT SCOTCH': 'Whiskey', 'STRAIGHT BOURBON WHISKIES': 'Whiskey',
    'STRAIGHT RYE WHISKIES': 'Whiskey', 'TENNESSEE WHISKIES': 'Whiskey',
    'BOTTLED IN BOND BOURBON': 'Whiskey',
    'SINGLE BARREL BOURBON WHISKIES': 'Whiskey',
    'WHISKEY LIQUEUR': 'Whiskey', 'ROCK & RYE': 'Whiskey',
    'BARBADOS RUM': 'Rum', 'FLAVORED RUM': 'Rum', 'JAMAICA RUM': 'Rum',
    'PUERTO RICO & VIRGIN ISLANDS RUM': 'Rum', 'SPICED RUM': 'Rum',
    'TEQUILA': 'Tequila',
    'AMERICAN DRY GINS': 'Gin', 'AMERICAN SLOE GINS': 'Gin',
    'FLAVORED GINS': 'Gin', 'IMPORTED DRY GINS': 'Gin',
    'AMERICAN GRAPE BRANDIES': 'Brandy', 'APRICOT BRANDIES': 'Brandy',
    'BLACKBERRY BRANDIES': 'Brandy', 'CHERRY BRANDIES': 'Brandy',
    'IMPORTED GRAPE BRANDIES': 'Brandy', 'MISCELLANEOUS BRANDIES': 'Brandy',
    'PEACH BRANDIES': 'Brandy',
    'APPLE SCHNAPPS': 'Schnapps', 'BUTTERSCOTCH SCHNAPPS': 'Schnapps',
    'CINNAMON SCHNAPPS': 'Schnapps', 'GRAPE SCHNAPPS': 'Schnapps',
    'IMPORTED SCHNAPPS': 'Schnapps', 'MISCELLANEOUS SCHNAPPS': 'Schnapps',
    'PEACH SCHNAPPS': 'Schnapps', 'PEPPERMINT SCHNAPPS': 'Schnapps',
    'RASPBERRY SCHNAPPS': 'Schnapps', 'ROOT BEER SCHNAPPS': 'Schnapps',
    'SPEARMINT SCHNAPPS': 'Schnapps', 'STRAWBERRY SCHNAPPS': 'Schnapps',
    'TROPICAL FRUIT SCHNAPPS': 'Schnapps', 'WATERMELON SCHNAPPS': 'Schnapps',
    'AMERICAN AMARETTO': 'Liqueurs', 'AMERICAN COCKTAILS': 'Liqueurs',
    'ANISETTE': 'Liqueurs', 'COFFEE LIQUEURS': 'Liqueurs',
    'CREAM LIQUEURS': 'Liqueurs', 'CREME DE ALMOND': 'Liqueurs',
    'DARK CREME DE CACAO': 'Liqueurs', 'GREEN CREME DE MENTHE': 'Liqueurs',
    'IMPORTED AMARETTO': 'Liqueurs',
    'MISC. AMERICAN CORDIALS & LIQUEURS': 'Liqueurs',
    'MISC. IMPORTED CORDIALS & LIQUEURS': 'Liqueurs',
    'TRIPLE SEC': 'Liqueurs', 'WHITE CREME DE CACAO': 'Liqueurs',
    'WHITE CREME DE MENTHE': 'Liqueurs',
    'AMERICAN ALCOHOL': 'Other', 'DECANTERS & SPECIALTY PACKAGES': 'Other',
    'DISTILLED SPIRITS SPECIALTY': 'Other',
}

df['category'] = df['Category Name'].map(cat_map)
print(f'Rows after category mapping: {len(df[df["category"].notna()]):,}')
print(f'Categories: {sorted(df["category"].dropna().unique())}')

## 2.2 Aggregate to Panel

**Unit of analysis:** county × year-month × category.

We build two panel datasets:
- **`panel`** — 8-category consolidation (main fixed-effects models)
- **`agg`** — granular 130 categories (deep-dive analysis in Section 7)

**Why liters per adult?** Normalizing by adult population (age 20+) makes cross-county comparisons fair.

In [ ]:
# Holiday/weekend days per month
holiday_per_month = (
    df[df['Is_Holiday_Weekend'] == 1]
    .groupby('ym')['date']
    .nunique()
    .reset_index(name='holiday_days')
)

# PANEL A: 8-category consolidated (main fixed-effects models)
df_cat = df.dropna(subset=['category'])

panel = (
    df_cat.groupby(['County', 'ym', 'category', 'year', 'month'])
    .agg(
        total_liters  = ('Volume Sold (Liters)', 'sum'),
        total_bottles = ('Bottles Sold', 'sum'),
        total_sales   = ('Sale (Dollars)', 'sum'),
        population    = ('Population', 'first'),
        weekend_count = ('is_weekend', 'sum'),
    )
    .reset_index()
)
panel['avg_price_liter'] = panel['total_sales'] / panel['total_liters']
panel = panel.merge(holiday_per_month, on='ym', how='left')
panel['holiday_days']     = panel['holiday_days'].fillna(0)
panel['liters_per_adult'] = panel['total_liters'] / panel['population']
panel = panel[(panel['liters_per_adult'] > 0) & (panel['avg_price_liter'] > 0)]
panel['log_liters_pa'] = np.log(panel['liters_per_adult'])
panel['log_price']     = np.log(panel['avg_price_liter'])
panel['log_price_sq']  = panel['log_price'] ** 2
panel['market_type']   = pd.cut(
    panel['population'],
    bins=[0, 30_000, 100_000, np.inf],
    labels=['Rural', 'Suburban', 'Urban']
).astype(str)
panel = panel.rename(columns={'County': 'county'})
panel = panel.sort_values(['county', 'category', 'ym']).reset_index(drop=True)
panel['log_liters_pa_lag'] = panel.groupby(['county', 'category'])['log_liters_pa'].shift(1)
panel['log_price_lag']     = panel.groupby(['county', 'category'])['log_price'].shift(1)
panel['year_c']    = panel['year'] - 2014
panel['month_str'] = panel['month'].astype(str).str.zfill(2)

# PANEL B: Granular 130-category (deep-dive analysis)
agg = (
    df.groupby(['County', 'Year', 'Month', 'Category Name'])
    .agg(
        Total_Liters  = ('Volume Sold (Liters)', 'sum'),
        Total_Sales   = ('Sale (Dollars)', 'sum'),
        Population    = ('Population', 'first'),
        Is_Holiday    = ('Is_Holiday_Weekend', 'max'),
        Holiday_Days  = ('Is_Holiday_Weekend', 'sum'),
        Weekend_Count = ('is_weekend', 'sum'),
    )
    .reset_index()
)
agg['Demand']     = agg['Total_Liters'] / agg['Population']
agg['Avg_Price']  = agg['Total_Sales'] / agg['Total_Liters']
agg['log_demand'] = np.log(agg['Demand'].replace(0, np.nan))
agg['log_price']  = np.log(agg['Avg_Price'].replace(0, np.nan))
agg = agg.dropna(subset=['log_demand', 'log_price'])

print(f'Panel (8-cat): {panel.shape}')
print(f'  {panel["county"].nunique()} counties | {panel["category"].nunique()} categories | {panel["ym"].nunique()} year-months')
print(f'Agg (granular): {agg.shape}')
print(f'  {agg["County"].nunique()} counties | {agg["Category Name"].nunique()} categories')
print(f'Obs with lag: {panel["log_liters_pa_lag"].notna().sum():,}')

---
# 3. Exploratory Data Analysis

## 3A. Price and Demand Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(agg['log_demand'], bins=60, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Log Demand (liters per adult)')
axes[0].set_xlabel('log(demand)')
sns.histplot(agg['log_price'], bins=60, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Distribution of Log Price (per liter)')
axes[1].set_xlabel('log(price per liter)')
plt.tight_layout()
plt.show()
print('Takeaway: Both variables are approximately log-normal, validating the log-log functional form.')

## 3B. Raw Log-Log Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(agg['log_price'], agg['log_demand'], alpha=0.12, s=8, color='steelblue')
m, b = np.polyfit(agg['log_price'], agg['log_demand'], 1)
x_range = np.linspace(agg['log_price'].min(), agg['log_price'].max(), 100)
ax.plot(x_range, m * x_range + b, color='red', linewidth=2, label=f'Raw slope = {m:.3f}')
ax.set_xlabel('log(price per liter)')
ax.set_ylabel('log(liters per adult)')
ax.set_title('Raw Log-Log Relationship: Price vs Demand')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Raw (unadjusted) elasticity: {m:.3f}')
print('Note: Fixed effects absorb confounds like county wealth — within-unit elasticity will differ.')

## 3C. Monthly Sales Trends by Category

In [ ]:
monthly = panel.groupby(['ym', 'category'])['total_liters'].sum().reset_index()
monthly['date'] = pd.to_datetime(monthly['ym'])

fig, ax = plt.subplots(figsize=(12, 5))
for cat, grp in monthly.groupby('category'):
    ax.plot(grp['date'], grp['total_liters'] / 1e6, label=cat, linewidth=1.3)
ax.set_title('Monthly Total Liters Sold by Category (2012-2017)', fontweight='bold')
ax.set_ylabel('Liters (millions)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
print('Takeaway: Vodka and Whiskey dominate volume. Year-end spikes confirm seasonal controls are essential.')

## 3D. Seasonality Heatmap — Avg Log Demand by Month x Category

Reveals which categories have the strongest seasonal patterns.

In [ ]:
top10_cats = agg.groupby('Category Name')['Total_Liters'].sum().nlargest(10).index
heat_data = (
    agg[agg['Category Name'].isin(top10_cats)]
    .groupby(['Month', 'Category Name'])['log_demand']
    .mean().unstack('Category Name')
)
heat_data.columns = [c[:22] for c in heat_data.columns]
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heat_data, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Avg log demand'})
ax.set_yticklabels(month_labels[:len(heat_data)], rotation=0)
ax.set_xlabel('Category')
ax.set_ylabel('Month')
ax.set_title('Seasonality Heatmap: Avg Log Demand by Month x Category')
plt.tight_layout()
plt.show()

## 3E. Price Distribution by Category

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = panel.groupby('category')['avg_price_liter'].median().sort_values().index
sns.boxplot(data=panel, y='category', x='avg_price_liter',
            order=order, flierprops=dict(marker='.', alpha=0.2), ax=ax)
ax.set_title('Price Per Liter Distribution by Category', fontweight='bold')
ax.set_xlabel('Average Price per Liter ($)')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()

## 3F. Per-Adult Demand by Market Type

In [ ]:
market_demand = panel.groupby(['market_type', 'category'])['liters_per_adult'].mean().reset_index()
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=market_demand, x='category', y='liters_per_adult', hue='market_type', ax=ax)
ax.set_title('Average Liters per Adult by Market Type and Category', fontweight='bold')
ax.set_ylabel('Liters per Adult')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Market Type')
plt.tight_layout()
plt.show()
print('Takeaway: Per-adult demand differs systematically by market type. Motivates M5 heterogeneity models.')

## 3G. Holiday vs Non-Holiday Demand

In [ ]:
panel['holiday_flag'] = np.where(panel['holiday_days'] > 0, 'Holiday/Weekend Month', 'Regular Month')
hol_demand = panel.groupby(['holiday_flag', 'category'])['liters_per_adult'].mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=hol_demand, x='category', y='liters_per_adult',
            hue='holiday_flag', palette=['steelblue', 'tomato'], ax=ax)
ax.set_title('Per-Adult Demand: Holiday/Weekend vs. Regular Months', fontweight='bold')
ax.set_ylabel('Liters per Adult')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='')
plt.tight_layout()
plt.show()

## 3H. Faceted Price vs Demand by Top Category

In [ ]:
top9 = agg.groupby('Category Name')['Total_Liters'].sum().nlargest(9).index.tolist()
sub9 = agg[agg['Category Name'].isin(top9)].copy()

g = sns.FacetGrid(sub9, col='Category Name', col_wrap=3, height=3.2, aspect=1.15, sharey=False)
g.map_dataframe(sns.regplot, x='log_price', y='log_demand',
                scatter_kws=dict(alpha=0.1, s=5, color='steelblue'),
                line_kws=dict(color='red', linewidth=1.5))
g.set_titles(col_template='{col_name}')
g.set_axis_labels('log(price)', 'log(demand)')
g.figure.suptitle('Raw Log-Log Price vs Demand by Top Category', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

---
# 4. Modeling Framework

## Why Log-Log?

In a **log-log** model, the coefficient on `log(price)` is directly interpretable as the **price elasticity of demand** — the % change in quantity for a 1% change in price.

## Fixed Effects Strategy

| Fixed Effect | What it controls for |
|---|---|
| County FE | Time-invariant county characteristics (demographics, income, culture) |
| Category FE | Permanent differences in demand level across spirit types |
| Month FE | Seasonal patterns common to all counties and categories |
| Year FE | Aggregate demand trends and macro shocks |

**`pyfixest` syntax:** `feols("y ~ x | fe1 + fe2 + fe3", data=df)` — variables after `|` are absorbed as fixed effects; standard errors clustered by county.

In [ ]:
def show_results(model, title='', stars=True):
    """Print key model results excluding fixed effect intercepts."""
    tidy = model.tidy()
    tidy = tidy[tidy.index != 'Intercept']

    def sig(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.10: return '*'
        return ''

    if stars:
        tidy['stars'] = tidy['Pr(>|t|)'].apply(sig)

    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(tidy[['Estimate', 'Std. Error', 'Pr(>|t|)', 'stars']].to_string())

    def _get(obj, *attrs, default='N/A'):
        for a in attrs:
            v = getattr(obj, a, None)
            if v is not None:
                return v
        return default

    n      = _get(model, '_N', 'N', 'nobs')
    r2     = _get(model, '_r2', 'r2')
    r2_adj = _get(model, '_r2_adj', 'adj_r2', 'r2_adj')

    n_str      = f'{int(n):,}'   if isinstance(n,      (int, float)) else str(n)
    r2_str     = f'{r2:.4f}'     if isinstance(r2,     (int, float)) else str(r2)
    r2_adj_str = f'{r2_adj:.4f}' if isinstance(r2_adj, (int, float)) else str(r2_adj)

    print(f"\n  N = {n_str}  |  R2 = {r2_str}  |  Adj. R2 = {r2_adj_str}")
    print(f"  Significance: * p<0.10  ** p<0.05  *** p<0.01")
    return tidy

---
# 5. Functional Form Models (M0 - M3)

These models test what **shape** best describes the price-demand relationship.

## M0 - Linear Baseline

Regresses raw `liters_per_adult` on price levels. Assumes a $1 increase always reduces demand by the same amount.

In [ ]:
m0 = pf.feols(
    'liters_per_adult ~ avg_price_liter + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m0, 'M0: Linear Baseline (DV = liters_per_adult)')

## M1 - Semi-Log

Logs the dependent variable but leaves price in levels.

In [ ]:
m1 = pf.feols(
    'log_liters_pa ~ avg_price_liter + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m1, 'M1: Semi-Log (DV = log(liters_per_adult), Price in levels)')

## M2 - Log-Log (Main Model)

Both demand and price are logged. The coefficient on `log_price` IS the price elasticity of demand.

In [ ]:
m2 = pf.feols(
    'log_liters_pa ~ log_price + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m2, 'M2: Log-Log Baseline (Coefficient on log_price = Price Elasticity)')

## M3 - Quadratic Log-Log

Adds (log P)^2 to test whether elasticity changes at different price levels.

In [ ]:
m3 = pf.feols(
    'log_liters_pa ~ log_price + log_price_sq + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m3, 'M3: Quadratic Log-Log (Tests Non-Constant Elasticity)')

b1 = m3.coef()['log_price']
b2 = m3.coef()['log_price_sq']
median_log_p = panel['log_price'].median()
print(f"\nImplied elasticity at median price (${np.exp(median_log_p):.2f}/L): {b1 + 2*b2*median_log_p:.4f}")

## Functional Form Comparison

In [ ]:
pf.etable(
    [m0, m1, m2, m3],
    labels={
        'liters_per_adult': 'Liters/Adult',
        'log_liters_pa': 'log(Liters/Adult)',
        'avg_price_liter': 'Price/Liter ($)',
        'log_price': 'log(Price/Liter)',
        'log_price_sq': 'log(Price/Liter)^2',
        'holiday_days': 'Holiday Days'
    },
    model_heads=['M0: Linear', 'M1: Semi-log', 'M2: Log-log', 'M3: Quadratic'],
    coef_fmt='b (se)',
    keep=['avg_price_liter', 'log_price', 'log_price_sq', 'holiday_days']
)

---
# 6. Heterogeneity - Broad Spirit Categories (M4 - M6)

## M4 - Category-Specific Elasticity (8 Broad Categories)

Estimates a separate price elasticity for each consolidated spirit category.

In [ ]:
m4 = pf.feols(
    'log_liters_pa ~ log_price:category + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m4, 'M4: Category-Specific Elasticity')

tidy_m4 = m4.tidy()
cat_coefs = tidy_m4[tidy_m4.index.str.contains('log_price:category')].copy()
cat_coefs['category'] = cat_coefs.index.str.replace('log_price:category::', '', regex=False)
cat_coefs = cat_coefs.sort_values('Estimate')

fig, ax = plt.subplots(figsize=(9, 5))
ax.axvline(-1, color='gray', linestyle='--', linewidth=1, label='Unit elastic (e=-1)')
ax.axvline(0, color='black', linewidth=0.5)
ax.barh(cat_coefs['category'], cat_coefs['Estimate'],
        xerr=1.96 * cat_coefs['Std. Error'],
        color=['tomato' if v < -1 else 'steelblue' for v in cat_coefs['Estimate']],
        capsize=4, alpha=0.8)
for _, row in cat_coefs.iterrows():
    ax.text(row['Estimate'] + 0.02, row['category'], f"{row['Estimate']:.3f}", va='center', fontsize=9)
ax.set_title('Price Elasticity by Category (M4)\nRed = Elastic (|e|>1), Blue = Inelastic', fontweight='bold')
ax.set_xlabel('Price Elasticity of Demand')
ax.legend()
plt.tight_layout()
plt.show()

## M5 - Urban vs. Rural Elasticity

In [ ]:
m5 = pf.feols(
    'log_liters_pa ~ log_price:market_type + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m5, 'M5: Urban vs. Rural Elasticity')

tidy_m5 = m5.tidy()
mkt_coefs = tidy_m5[tidy_m5.index.str.contains('log_price:market_type')].copy()
mkt_coefs['market'] = mkt_coefs.index.str.replace('log_price:market_type::', '', regex=False)

fig, ax = plt.subplots(figsize=(7, 4))
colors_mkt = {'Rural': 'seagreen', 'Suburban': 'steelblue', 'Urban': 'tomato'}
for _, row in mkt_coefs.iterrows():
    c = colors_mkt.get(row['market'], 'gray')
    ax.bar(row['market'], row['Estimate'], yerr=1.96 * row['Std. Error'], color=c, alpha=0.8, capsize=5)
    ax.text(row['market'], row['Estimate'] - 0.03, f"{row['Estimate']:.3f}",
            ha='center', va='top', fontweight='bold')
ax.axhline(-1, color='gray', linestyle='--', label='Unit elastic (e=-1)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Price Elasticity by Market Type (M5)', fontweight='bold')
ax.set_ylabel('Price Elasticity')
ax.legend()
plt.tight_layout()
plt.show()

## M6 - Category x Market Type Interaction

In [ ]:
m6 = pf.feols(
    'log_liters_pa ~ log_price:category:market_type + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)

tidy_m6 = m6.tidy()
interaction_coefs = tidy_m6[tidy_m6.index.str.startswith('log_price:')].copy()

def extract_cat(s):
    match = re.search(r'category\[([^\]]+)\]', str(s))
    return match.group(1) if match else None

def extract_mkt(s):
    match = re.search(r'market_type\[([^\]]+)\]', str(s))
    return match.group(1) if match else None

interaction_coefs['category'] = [extract_cat(i) for i in interaction_coefs.index]
interaction_coefs['market']   = [extract_mkt(i) for i in interaction_coefs.index]
interaction_coefs = interaction_coefs.dropna(subset=['category', 'market'])

pivot = interaction_coefs.pivot_table(index='category', columns='market', values='Estimate', aggfunc='mean')
fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind='bar', ax=ax, alpha=0.85)
ax.axhline(-1, color='gray', linestyle='--', linewidth=1, label='Unit elastic (e=-1)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Price Elasticity by Category and Market Type (M6)', fontweight='bold')
ax.set_ylabel('Price Elasticity')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Market Type')
plt.tight_layout()
plt.show()

---
# 7. Heterogeneity - Granular Analysis (M4b, M5b, M5c)

These models retain all 130 granular categories and all 99 counties.

## M4b - Granular Category Elasticity (130 Categories)

Separate OLS per category. Reveals ultra-elastic niche items invisible in the consolidated model.

In [ ]:
cat_elas = []
for cat, grp in agg.groupby('Category Name'):
    grp = grp.dropna(subset=['log_price', 'log_demand'])
    if len(grp) < 30:
        continue
    X_c = sm.add_constant(grp['log_price'])
    res = sm.OLS(grp['log_demand'], X_c).fit()
    cat_elas.append({
        'Category':   cat,
        'Elasticity': res.params['log_price'],
        'p_value':    res.pvalues['log_price'],
        'R2':         res.rsquared,
        'N':          len(grp),
    })

cat_df = pd.DataFrame(cat_elas).sort_values('Elasticity')

fig, ax = plt.subplots(figsize=(11, max(5, len(cat_df) * 0.3)))
colors_bar = ['tomato' if e < 0 else 'steelblue' for e in cat_df['Elasticity']]
ax.barh(cat_df['Category'], cat_df['Elasticity'], color=colors_bar)
for i, (_, row) in enumerate(cat_df.iterrows()):
    star = '***' if row['p_value'] < 0.001 else ('**' if row['p_value'] < 0.01 else ('*' if row['p_value'] < 0.05 else ''))
    offset = 0.02 if row['Elasticity'] >= 0 else -0.02
    ax.text(row['Elasticity'] + offset, i, star, va='center', fontsize=7,
            ha='left' if row['Elasticity'] >= 0 else 'right')
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(-1, color='gray', linewidth=0.8, linestyle='--', label='|b|=1 (unit elastic)')
ax.set_xlabel('Price Elasticity')
ax.set_title('Price Elasticity by Granular Liquor Category\n* p<0.05  ** p<0.01  *** p<0.001')
ax.legend()
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.show()

print('\nTop 5 most price-sensitive categories:')
print(cat_df.head(5)[['Category','Elasticity','p_value','R2','N']].to_string(index=False))

## M5b - County-Level Price Sensitivity

In [ ]:
county_elas = []
for county, grp in agg.groupby('County'):
    grp = grp.dropna(subset=['log_price', 'log_demand'])
    if len(grp) < 30:
        continue
    X_c = sm.add_constant(grp['log_price'])
    res = sm.OLS(grp['log_demand'], X_c).fit()
    county_elas.append({
        'County':     county,
        'Elasticity': res.params['log_price'],
        'p_value':    res.pvalues['log_price'],
        'R2':         res.rsquared,
        'N':          len(grp),
    })

county_df = pd.DataFrame(county_elas).sort_values('Elasticity')

fig, ax = plt.subplots(figsize=(11, max(5, len(county_df) * 0.28)))
colors_bar = ['tomato' if e < 0 else 'steelblue' for e in county_df['Elasticity']]
ax.barh(county_df['County'], county_df['Elasticity'], color=colors_bar)
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(-1, color='gray', linewidth=0.8, linestyle='--', label='|b|=1')
ax.set_xlabel('Price Elasticity')
ax.set_title('Price Elasticity by County (OLS per county)')
ax.tick_params(axis='y', labelsize=7)
ax.legend()
plt.tight_layout()
plt.show()

print('\nMost price-sensitive counties:')
print(county_df.head(5)[['County','Elasticity','p_value','R2','N']].to_string(index=False))

## M5c - Price Elasticity Heatmap: Top Counties x Top Categories

In [ ]:
top8_counties = agg['County'].value_counts().head(8).index.tolist()
top8_cats     = agg['Category Name'].value_counts().head(8).index.tolist()

heat_elas = {}
for cat in top8_cats:
    row = {}
    for county in top8_counties:
        sub = agg[(agg['County'] == county) & (agg['Category Name'] == cat)]
        sub = sub.dropna(subset=['log_price', 'log_demand'])
        if len(sub) < 10:
            row[county] = np.nan
        else:
            res = sm.OLS(sub['log_demand'], sm.add_constant(sub['log_price'])).fit()
            row[county] = res.params['log_price']
    heat_elas[cat[:22]] = row

heat_df = pd.DataFrame(heat_elas).T

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heat_df, cmap='RdYlGn', center=0, annot=True, fmt='.2f',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Price Elasticity'})
ax.set_title('Price Elasticity Heatmap: Top Categories x Top Counties')
ax.set_xlabel('County')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()
print('Takeaway: Red cells are most elastic - highest risk from price increases, best for volume promotions.')

---
# 8. Calendar and Timing Models (M7 - M9)

## M7 - Holiday x Price Interaction

Tests whether price sensitivity changes during holiday/weekend-heavy months.

In [ ]:
m7 = pf.feols(
    'log_liters_pa ~ log_price + holiday_days + log_price:holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m7, 'M7: Holiday x Price Interaction')

b_price    = m7.coef()['log_price']
b_interact = m7.coef()['log_price:holiday_days']
hol_range  = np.arange(0, 11)
marg_elas  = b_price + b_interact * hol_range

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hol_range, marg_elas, 'o-', color='steelblue', linewidth=2, markersize=7)
ax.axhline(-1, color='gray', linestyle='--', label='Unit elastic (e=-1)')
ax.axhline(0, color='black', linewidth=0.5)
for x, y in zip(hol_range, marg_elas):
    ax.text(x, y + 0.01, f'{y:.3f}', ha='center', fontsize=8.5)
ax.set_title('Marginal Price Elasticity by Holiday/Weekend Days in Month (M7)', fontweight='bold')
ax.set_xlabel('Holiday/Weekend Days in Month')
ax.set_ylabel('Marginal Price Elasticity')
ax.set_xticks(hol_range)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nElasticity with 0 holiday days:  {b_price:.4f}")
print(f"Elasticity with 10 holiday days: {b_price + b_interact*10:.4f}")
print("\nConclusion: Consumers are less price-sensitive during holiday months.")
print("Optimal strategy: HOLD prices during holidays; reserve discounts for January-March.")

## M7b - Marginal Elasticity Across Weekend Activity

Visualises how price sensitivity varies with weekend activity for holiday vs non-holiday periods.

In [ ]:
interact_df = agg.dropna(subset=['log_demand', 'log_price']).copy()
interact_df['price_x_holiday'] = interact_df['log_price'] * interact_df['Is_Holiday']
interact_df['price_x_weekend'] = interact_df['log_price'] * interact_df['Weekend_Count']

X_int = sm.add_constant(interact_df[[
    'log_price', 'Is_Holiday', 'price_x_holiday', 'Weekend_Count', 'price_x_weekend'
]])
model_int = sm.OLS(interact_df['log_demand'], X_int).fit(cov_type='HC3')

wk_range = np.linspace(0, agg['Weekend_Count'].quantile(0.95), 50)
elas_nonhol = model_int.params['log_price'] + model_int.params['price_x_weekend'] * wk_range
elas_hol    = (model_int.params['log_price'] + model_int.params['price_x_holiday']
               + model_int.params['price_x_weekend'] * wk_range)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(wk_range, elas_nonhol, color='steelblue', linewidth=2, label='Non-Holiday')
ax.plot(wk_range, elas_hol,    color='coral', linewidth=2, linestyle='--', label='Holiday')
ax.axhline(0,  color='black', linewidth=0.7)
ax.axhline(-1, color='gray',  linewidth=0.7, linestyle=':', label='Unit elastic')
ax.set_xlabel('Weekend Transaction Count')
ax.set_ylabel('Marginal Price Elasticity')
ax.set_title('Price Sensitivity vs Weekend Activity and Holiday Status (M7b)')
ax.legend()
plt.tight_layout()
plt.show()

## M8 - Month x Price Interaction (Seasonal Elasticity)

In [ ]:
m8 = pf.feols(
    'log_liters_pa ~ log_price:month_str + holiday_days | county + category + month + year',
    data=panel, vcov={'CRV1': 'county'}
)

tidy_m8 = m8.tidy()
month_coefs = tidy_m8[tidy_m8.index.str.contains('log_price:month_str')].copy()

def extract_month(s):
    match = re.search(r'(\d+)', str(s).replace('log_price', ''))
    return int(match.group(1)) if match else None

month_coefs['month_n'] = [extract_month(i) for i in month_coefs.index]
month_coefs = month_coefs.dropna(subset=['month_n'])
month_coefs['month_n'] = month_coefs['month_n'].astype(int)
month_coefs = month_coefs.sort_values('month_n')

month_labels_list = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_coefs['month_label'] = month_coefs['month_n'].apply(lambda x: month_labels_list[x-1])

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(month_coefs['month_n'],
                month_coefs['Estimate'] - 1.96*month_coefs['Std. Error'],
                month_coefs['Estimate'] + 1.96*month_coefs['Std. Error'],
                alpha=0.15, color='steelblue')
ax.plot(month_coefs['month_n'], month_coefs['Estimate'],
        'o-', color='steelblue', linewidth=2, markersize=6)
ax.axhline(-1, color='gray', linestyle='--', label='Unit elastic (e=-1)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(month_coefs['month_n'])
ax.set_xticklabels(month_coefs['month_label'])
ax.set_title('Price Elasticity by Calendar Month (M8)\nShaded = 95% CI', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Price Elasticity')
ax.legend()
plt.tight_layout()
plt.show()

most_elastic  = month_coefs.loc[month_coefs['Estimate'].idxmin(), 'month_label']
least_elastic = month_coefs.loc[month_coefs['Estimate'].idxmax(), 'month_label']
print(f"Most elastic month:   {most_elastic}  ({month_coefs['Estimate'].min():.4f})")
print(f"Least elastic month:  {least_elastic} ({month_coefs['Estimate'].max():.4f})")

## M9 - Year Trend in Elasticity

In [ ]:
m9 = pf.feols(
    'log_liters_pa ~ log_price + log_price:year_c + holiday_days | county + category + month',
    data=panel, vcov={'CRV1': 'county'}
)
show_results(m9, 'M9: Year Trend in Elasticity (year FE dropped to avoid collinearity)')

b9_p   = m9.coef()['log_price']
b9_int = m9.coef()['log_price:year_c']
years       = np.arange(2012, 2018)
year_c_vals = years - 2014
elasticities = b9_p + b9_int * year_c_vals

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(years, elasticities, 'o-', color='tomato', linewidth=2, markersize=8)
ax.axhline(-1, color='gray', linestyle='--', label='Unit elastic (e=-1)')
ax.axhline(0, color='black', linewidth=0.5)
for x, y in zip(years, elasticities):
    ax.text(x, y - 0.015, f'{y:.3f}', ha='center', va='top', fontsize=9)
ax.set_title('Estimated Price Elasticity Trend by Year (M9)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Price Elasticity')
ax.set_xticks(years)
ax.legend()
plt.tight_layout()
plt.show()

---
# 9. Dynamic Demand Models (M10 - M11)

## M10 - Lagged Demand (Habit Persistence)

- **Short-run elasticity** = beta_1 (immediate demand response)
- **Long-run elasticity** = beta_1 / (1 - beta_2) (full effect after habits adjust)

In [ ]:
panel_lag = panel.dropna(subset=['log_liters_pa_lag'])

m10 = pf.feols(
    'log_liters_pa ~ log_price + log_liters_pa_lag + holiday_days | county + category + month + year',
    data=panel_lag, vcov={'CRV1': 'county'}
)
show_results(m10, 'M10: Lagged Demand - Habit Persistence')

sr_elas  = m10.coef()['log_price']
lag_coef = m10.coef()['log_liters_pa_lag']
lr_elas  = sr_elas / (1 - lag_coef)

print('\n' + '='*50)
print('  Dynamic Elasticity Decomposition')
print('='*50)
print(f'  Short-run elasticity (b1):           {sr_elas:.4f}')
print(f'  Lag coefficient (b2):                {lag_coef:.4f}')
print(f'  Long-run elasticity (b1 / (1-b2)):  {lr_elas:.4f}')
print(f'\n  Long-run effect is {abs(lr_elas/sr_elas):.1f}x larger than short-run')
print(f'  Demand shows {"strong" if lag_coef > 0.3 else "moderate"} month-to-month persistence')

## M11 - Distributed Price Lag

In [ ]:
panel_distlag = panel.dropna(subset=['log_price_lag', 'log_liters_pa_lag'])

m11 = pf.feols(
    'log_liters_pa ~ log_price + log_price_lag + holiday_days | county + category + month + year',
    data=panel_distlag, vcov={'CRV1': 'county'}
)
show_results(m11, 'M11: Distributed Price Lag')

b_cur = m11.coef()['log_price']
b_lag = m11.coef()['log_price_lag']
print(f'\nContemporaneous elasticity (b1): {b_cur:.4f}')
print(f'Lagged price elasticity (b2):    {b_lag:.4f}')
print(f'Total effect (b1 + b2):          {b_cur + b_lag:.4f}')
print('\nIf b2 is not significant: price effects are immediate with no echo month-over-month.')

---
# 10. Cross-Price Elasticity (M12)

Tests whether spirits are substitutes. Positive cross-price coefficient = substitutes.

In [ ]:
cross_prices = (
    panel[panel['category'].isin(['Vodka', 'Whiskey'])]
    .groupby(['county', 'ym', 'category'])
    .apply(lambda g: np.average(g['avg_price_liter'], weights=g['total_liters']))
    .reset_index(name='avg_price')
    .pivot(index=['county', 'ym'], columns='category', values='avg_price')
    .reset_index()
    .rename(columns={'Vodka': 'price_vodka', 'Whiskey': 'price_whiskey'})
)
cross_prices['log_price_vodka']   = np.log(cross_prices['price_vodka'])
cross_prices['log_price_whiskey'] = np.log(cross_prices['price_whiskey'])

panel_vw = (
    panel[panel['category'].isin(['Vodka', 'Whiskey'])]
    .merge(cross_prices, on=['county', 'ym'], how='inner')
)

m12_vodka = pf.feols(
    'log_liters_pa ~ log_price_vodka + log_price_whiskey + holiday_days | county + month + year',
    data=panel_vw[panel_vw['category'] == 'Vodka'],
    vcov={'CRV1': 'county'}
)
m12_whiskey = pf.feols(
    'log_liters_pa ~ log_price_whiskey + log_price_vodka + holiday_days | county + month + year',
    data=panel_vw[panel_vw['category'] == 'Whiskey'],
    vcov={'CRV1': 'county'}
)

print('--- VODKA DEMAND ---')
show_results(m12_vodka, 'M12a: Vodka Demand (own + cross-price)')
print('\n--- WHISKEY DEMAND ---')
show_results(m12_whiskey, 'M12b: Whiskey Demand (own + cross-price)')

pf.etable(
    [m12_vodka, m12_whiskey],
    labels={
        'log_liters_pa': 'log(Liters/Adult)',
        'log_price_vodka': 'log(Vodka Price)',
        'log_price_whiskey': 'log(Whiskey Price)',
        'holiday_days': 'Holiday Days'
    },
    model_heads=['Vodka Demand', 'Whiskey Demand'],
    coef_fmt='b (se)',
    keep=['log_price_vodka', 'log_price_whiskey', 'holiday_days']
)

---
# 11. Model Comparison (AIC / BIC / Adj. R2)

In [ ]:
cmp = agg.sort_values(['County', 'Category Name', 'Year', 'Month']).copy()
cmp['lag_log_demand'] = cmp.groupby(['County', 'Category Name'])['log_demand'].shift(1)
cmp = cmp.dropna(subset=['log_demand', 'log_price', 'lag_log_demand'])
cmp['price_x_holiday'] = cmp['log_price'] * cmp['Is_Holiday']
cmp['price_x_weekend'] = cmp['log_price'] * cmp['Weekend_Count']
y_cmp = cmp['log_demand']

def fit_cmp(cols, label):
    res = sm.OLS(y_cmp, sm.add_constant(cmp[cols].astype(float))).fit()
    return {'Model': label, 'R2': round(res.rsquared,4),
            'Adj. R2': round(res.rsquared_adj,4),
            'AIC': round(res.aic,1), 'BIC': round(res.bic,1), 'N': int(res.nobs)}

comp_df = pd.DataFrame([
    fit_cmp(['log_price'],                                                    'M0: Price only'),
    fit_cmp(['log_price', 'Is_Holiday'],                                      'M1: + Holiday'),
    fit_cmp(['log_price', 'Is_Holiday', 'Weekend_Count'],                     'M2: + Weekend'),
    fit_cmp(['log_price', 'Is_Holiday', 'Weekend_Count',
             'price_x_holiday', 'price_x_weekend'],                           'M3: + Interactions'),
    fit_cmp(['log_price', 'Is_Holiday', 'Weekend_Count', 'lag_log_demand'],   'M4: + Lagged demand'),
])
print(comp_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(comp_df['Model'], comp_df['Adj. R2'], color='steelblue')
axes[0].set_xlabel('Adjusted R2  (higher is better)')
axes[0].set_title('Model Comparison - Adjusted R2')
for i, v in enumerate(comp_df['Adj. R2']):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
axes[1].barh(comp_df['Model'], comp_df['AIC'], color='coral')
axes[1].set_xlabel('AIC  (lower is better)')
axes[1].set_title('Model Comparison - AIC')
for i, v in enumerate(comp_df['AIC']):
    axes[1].text(v + comp_df['AIC'].max()*0.003, i, f'{v:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Full Fixed-Effects Model Comparison Table

In [ ]:
pf.etable(
    [m0, m1, m2, m3, m7, m9, m10, m11],
    labels={
        'liters_per_adult': 'Liters/Adult',
        'log_liters_pa': 'log(Liters/Adult)',
        'avg_price_liter': 'Price/Liter ($)',
        'log_price': 'log(Price/Liter)',
        'log_price_sq': 'log(Price)^2',
        'holiday_days': 'Holiday Days',
        'log_price:holiday_days': 'log(Price) x Holiday',
        'log_price:year_c': 'log(Price) x Year',
        'log_liters_pa_lag': 'log(Demand t-1)',
        'log_price_lag': 'log(Price t-1)',
    },
    model_heads=['M0\nLinear', 'M1\nSemi-log', 'M2\nLog-log',
                 'M3\nQuadratic', 'M7\nHoliday', 'M9\nYear Trend',
                 'M10\nLag Demand', 'M11\nLag Price'],
    coef_fmt='b (se)',
    keep=['avg_price_liter', 'log_price', 'log_price_sq', 'holiday_days',
          'log_price:holiday_days', 'log_price:year_c',
          'log_liters_pa_lag', 'log_price_lag']
)

---
# 12. Holdout Validation (2012-2015 Train / 2016-2017 Test)

Validates the best model's predictive accuracy on an unseen time period.

In [ ]:
val = agg.sort_values(['County', 'Category Name', 'Year', 'Month']).copy()
val['lag_log_demand'] = val.groupby(['County', 'Category Name'])['log_demand'].shift(1)
val = val.dropna(subset=['log_demand', 'log_price', 'lag_log_demand'])

train = val[val['Year'] <= 2015]
test  = val[val['Year'] >= 2016]

feats = ['log_price', 'Is_Holiday', 'Weekend_Count', 'lag_log_demand']
X_tr = sm.add_constant(train[feats].astype(float))
X_te = sm.add_constant(test[feats].astype(float)).reindex(columns=X_tr.columns, fill_value=0)

model_val = sm.OLS(train['log_demand'], X_tr).fit()
y_pred    = model_val.predict(X_te)
y_true    = test['log_demand']

rmse_train = np.sqrt(mean_squared_error(train['log_demand'], model_val.fittedvalues))
rmse_test  = np.sqrt(mean_squared_error(y_true, y_pred))
r2_test    = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_true.mean())**2)

print(f'Train years: 2012-2015  ({len(train):,} obs)')
print(f'Test  years: 2016-2017  ({len(test):,} obs)')
print(f'Train RMSE: {rmse_train:.4f}')
print(f'Test  RMSE: {rmse_test:.4f}')
print(f'R2  (test): {r2_test:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(y_pred, y_true, alpha=0.15, s=8, color='steelblue')
mn = min(y_pred.min(), y_true.min())
mx = max(y_pred.max(), y_true.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Predicted log demand')
axes[0].set_ylabel('Actual log demand')
axes[0].set_title(f'Holdout: Predicted vs Actual  (RMSE = {rmse_test:.4f})')
axes[0].legend()
resid_test = y_true.values - y_pred.values
sns.histplot(resid_test, bins=50, kde=True, ax=axes[1], color='coral')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Prediction Error (actual - predicted)')
axes[1].set_title('Holdout Prediction Error Distribution')
plt.tight_layout()
plt.show()

---
# 13. Conclusions and Managerial Implications

## Summary of Findings

| Research Question | Model | Finding |
|---|---|---|
| Is overall demand elastic or inelastic? | M2 | **Inelastic (e approx -0.46)** - market is not very price sensitive overall |
| Best functional form? | M0-M3 | **Log-log (M2)** - linear fails; quadratic confirms non-constant elasticity |
| Most price-sensitive broad category? | M4 | **Whiskey (e approx -1.30)** - only consolidated category crossing unit elastic |
| Most price-sensitive granular items? | M4b | **Specialty schnapps and flavored spirits (e < -2)** - invisible in 8-category view |
| Urban vs rural differences? | M5 | **Urban consumers NOT significantly price sensitive (p approx 0.75)**; rural and suburban drive all aggregate elasticity |
| Which counties to target? | M5b/M5c | **Mills, Fremont, Henry** counties most price-sensitive; cross-referencing with category identifies highest-ROI promotion targets |
| Less sensitive on holidays? | M7 | **Yes** - elasticity softens from -0.48 (no holidays) to -0.15 (10 holiday days) |
| Seasonal elasticity cycle? | M8 | **Yes** - January most elastic; October least elastic |
| Stability over time? | M9 | **No significant trend** - elasticity stable 2012-2017 |
| Habit persistence? | M10 | **Strong** (lag b approx 0.45***). Long-run e approx 1.8x larger than short-run |
| Lagged price effects? | M11 | **No** - price effects are immediate; no echo month-over-month |
| Vodka-Whiskey substitution? | M12 | **Yes** - cross-price e approx +0.30-0.41***; coordinated pricing required |
| Out-of-sample accuracy? | Holdout | **Test R2 approx 0.75, RMSE approx 0.85** on 2016-2017 |

---

## Managerial Recommendations

**1. Price Whiskey carefully - it is the only revenue-at-risk broad category.**
Whiskey demand is elastic, meaning a price increase loses more in volume than it gains in margin. All other consolidated categories can absorb moderate price increases.

**2. Do not discount during the holiday season - hold prices instead.**
Holiday months are when consumers are least price-sensitive. Discounting in Nov-Dec wastes margin. Save promotions for Jan-Mar.

**3. Build separate pricing rules for rural vs. urban counties.**
Urban demand is statistically insensitive to price. Rural and suburban counties drive the entire aggregate elasticity.

**4. Use the county x category heatmap (M5c) to prioritize promotion budgets.**
Cells showing the most negative elasticity are the highest-return targets for promotions.

**5. Plan for long-run price effects - they are ~1.8x larger than month-1 data shows.**
The habit persistence result (M10) means demand keeps adjusting for months after a price change.

**6. Coordinate Vodka and Whiskey pricing jointly.**
Cross-price elasticity of ~0.3-0.4 between the two largest categories means independent pricing leaves revenue on the table.

**7. Watch specialty schnapps and flavored spirits for volume-sensitive promotions.**
The granular analysis (M4b) reveals niche categories with elasticities exceeding -4. These items respond dramatically to price promotions.

---

*Analysis conducted using Python with `pyfixest` for panel fixed-effects models and `statsmodels` for granular OLS and holdout validation. Two panel datasets: (1) ~49,000 observations across 99 counties, 8 categories, 56 year-months with county + category + month + year fixed effects; (2) granular 130-category panel for deep-dive OLS. Standard errors clustered by county.*